# ব্যয় দাবির বিশ্লেষণ

এই নোটবুকটি দেখায় কীভাবে প্লাগইন ব্যবহার করে এজেন্ট তৈরি করা যায় যা স্থানীয় রসিদ চিত্র থেকে ভ্রমণ ব্যয় প্রক্রিয়া করে, একটি ব্যয় দাবি ইমেল তৈরি করে, এবং পাই চার্টের মাধ্যমে ব্যয় তথ্য দৃশ্যায়ন করে। এজেন্টরা কাজের প্রেক্ষাপট অনুযায়ী ডায়নামিকভাবে ফাংশন নির্বাচন করে।

ধাপসমূহ:
1. OCR এজেন্ট স্থানীয় রসিদ চিত্র প্রক্রিয়া করে এবং ভ্রমণ ব্যয় তথ্য নিষ্কাশন করে।
2. ইমেল এজেন্ট একটি ব্যয় দাবি ইমেল তৈরি করে।

### ভ্রমণ ব্যয়ের একটি উদাহরণ পরিস্থিতি:
কল্পনা করুন আপনি একটি ব্যবসায়িক সভার জন্য অন্য একটি শহরে ভ্রমণ করছেন। আপনার কোম্পানির একটি নীতি আছে যা সব যুক্তিসংগত ভ্রমণ-সম্পর্কিত ব্যয় ফেরত দেবে। এখানে সম্ভাব্য ভ্রমণ ব্যয়ের একটি তালিকা:
- পরিবহন:
আপনার বাস শহর থেকে গন্তব্য শহরে রাউন্ড ট্রিপের বিমান টিকিট।
এয়ারপোর্ট থেকে যাতায়াতের জন্য ট্যাক্সি বা রাইড-হেলিং সেবা।
গন্তব্য শহরের স্থানীয় পরিবহন (যেমন পাবলিক ট্রানজিট, ভাড়া করা গাড়ি, বা ট্যাক্সি)।

- আবাসন:
সভার স্থান কাছাকাছি একটি মধ্যম মানের ব্যবসায়িক হোটেলে তিন রাতের হোটেল থাকা।

- খাবার:
প্রতিদিনের খাবারের ভাতা ব্রেকফাস্ট, লাঞ্চ এবং ডিনারের জন্য, কোম্পানির দৈনিক ভাতা নীতির উপর ভিত্তি করে।

- বিভিন্ন ব্যয়:
এয়ারপোর্টে পার্কিং ফি।
হোটেলে ইন্টারনেট অ্যাক্সেস চার্জ।
টিপ বা ছোট সার্ভিস চার্জ।

- ডকুমেন্টেশন:
আপনি সব রসিদ (ফ্লাইট, ট্যাক্সি, হোটেল, খাবার ইত্যাদি) এবং একটি পূর্ণাঙ্গ ব্যয় রিপোর্ট জমা দেন ফেরতের জন্য।


## প্রয়োজনীয় লাইব্রেরি ইমপোর্ট করুন

নোটবুকের জন্য প্রয়োজনীয় লাইব্রেরি এবং মডিউলগুলি ইমপোর্ট করুন।


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## ব্যয় মডেল সংজ্ঞায়িত করুন

একক ব্যয়ের জন্য একটি Pydantic মডেল তৈরি করুন এবং একটি ExpenseFormatter ক্লাস তৈরি করুন যা ব্যবহারকারীর প্রশ্নকে কাঠামোবদ্ধ ব্যয় তথ্য হিসেবে রূপান্তর করবে।

প্রত্যেকটি ব্যয় নিম্নলিখিত ফরম্যাটে উপস্থাপন করা হবে:
`{'date': '07-Mar-2025', 'description': 'গন্তব্যস্থলে ফ্লাইট', 'amount': 675.99, 'category': 'পরিবহন'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## টুল সংজ্ঞায়িত করা - ইমেল তৈরি করা

একটা টুল ফাংশন তৈরি করুন যা খরচের দাবি জমা দেওয়ার জন্য একটি ইমেল তৈরি করবে।  
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।  
- এটি খরচের মোট পরিমাণ গণনা করে এবং বিবরণগুলি একটি ইমেল বডিতে ফরম্যাট করে।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Tool for Extracting Travel Expenses from Receipt Images

Create a tool function to extract travel expenses from receipt images.
- এই টুলটি Microsoft Agent Framework এর `@tool` ডেকোরেটর ব্যবহার করে।
- এটি রসিদের ছবি পড়ে, সেটিকে base64 হিসাবে এনকোড করে, এবং এজেন্টের বিশ্লেষণের জন্য ডেটা URI রিটার্ন করে।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processing Expenses

`WorkflowBuilder` ব্যবহার করে এজেন্টগুলি সংজ্ঞায়িত করুন এবং তাদের একটি ধারাবাহিক ওয়ার্কফ্লোতে সংযুক্ত করুন।
- OCR এজেন্ট রসিদ চিত্র থেকে গঠনমূলক খরচের তথ্য নিষ্কাশন করে `load_receipt_image` টুল ব্যবহার করে।
- ইমেইল এজেন্ট নিষ্কাশিত তথ্য নিয়ে একটি পেশাদার খরচ দাবি ইমেইল তৈরি করে `generate_expense_email` টুল ব্যবহার করে।
- `add_edge` সহ `WorkflowBuilder` একটি ধারাবাহিক পাইপলাইন তৈরি করে: OCR Agent → Email Agent।


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

ক্রমাগত কার্যপ্রবাহ তৈরি করুন এবং চালান যাতে রশিদের ছবি প্রক্রিয়া করা যায় এবং খরচ দাবি ইমেইল তৈরি করা যায়।


> **নোট:** এই ওয়ার্কফ্লোটি বর্তমানে রশিদের ছবি বেস64 টেক্সট হিসেবে পাঠায়, যা বেশির ভাগ চ্যাট মডেল (gpt-4o সহ) ছবি হিসেবে বিবেচনা করবে না।
> এটি মডেলের কনটেক্সট উইন্ডোও অধিকতর করতে পারে। Azure AI Vision (অথবা অন্য কোনো OCR টুল) ব্যবহার করে OCR চালানো পছন্দ করুন এবং শুধুমাত্র উত্তোলিত টেক্সট পাঠান, অথবা ছবিটি একটি `image_url` মেসেজ হিসেবে পাঠানোর জন্য পুনর্গঠন করুন।
> যদি আপনি কেবল কনটেক্সট ত্রুটি এড়াতে চান, তাহলে একটি ছোট রশিদের ছবি বা বড় কনটেক্সট উইন্ডো সহ মডেল চেষ্টা করুন।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
